# All of annplyr

This notebook is a compact tour of `annplyr`. It shows how the `adata.ap` accessor works across AnnData metadata, feature matrices, layers, embeddings, and grouped summaries.

## Set up Data

Importing `annplyr` registers the `ap` accessor on AnnData objects.

In [ ]:
# /// script
# requires-python = ">=3.12"
# dependencies = [
#     "anndata",
#     "annplyr",
#     "numpy",
#     "pandas",
# ]
# ///

import anndata as ad
import numpy as np
import pandas as pd
import annplyr as ap

In [ ]:
x = np.array(
    [
        [1, 0, 5, 2],
        [2, 3, 0, 1],
        [0, 1, 4, 0],
        [5, 2, 3, 8],
        [3, 0, 1, 4],
    ],
    dtype=float,
)

obs = pd.DataFrame(
    {
        "batch": ["A", "A", "B", "B", "A"],
        "score": [1.0, 2.0, 0.5, 3.0, 2.5],
        "cell_type": ["T", "B", "T", "Mono", "B"],
    },
    index=pd.Index([f"c{i}" for i in range(5)], name="cell"),
)

var = pd.DataFrame(
    {
        "feature_type": ["rna", "rna", "protein", "rna"],
        "chrom": ["chr1", "chr2", "chr1", "chr3"],
        "length": [100, 200, 150, 80],
    },
    index=pd.Index([f"g{i}" for i in range(4)], name="gene"),
)

adata = ad.AnnData(X=x, obs=obs, var=var)
adata.layers["counts"] = x * 10
adata.obsm["X_pca"] = np.array(
    [[0.1, 1.0], [-0.2, 0.0], [0.3, -1.0], [1.0, 2.0], [0.5, 1.5]],
    dtype=float,
)
adata.varm["loadings"] = np.array(
    [[0.5, 2.0], [-0.1, 3.0], [0.3, 1.0], [1.2, 0.0]],
    dtype=float,
)

adata

## Filter

`filter` subsets observations and variables. Predicates can read from `obs`, `var`, `X`, layers, `obsm`, `varm`, or the observation and variable names.

In [ ]:
adata.ap.filter(obs=ap.col("batch") == "A")

That is equivalent to standard AnnData indexing:

In [ ]:
adata[adata.obs["batch"] == "A", :]

In [ ]:
adata.ap.filter(
    obs=(ap.col("batch") == "A", ap.col("score") >= 2),
    var=ap.col("feature_type") == "rna",
    x=ap.col("g0") > 10,
    obsm={"X_pca": ap.col("0") >= 0},
    layer="counts",
)

In [ ]:
adata.ap.filter(
    obs_names=ap.obs_names.str.starts_with("c"),
    var_names=ap.var_names.str.starts_with("g"),
)

## Select

`select` keeps metadata columns and matrix features. Tidyselect-style helpers make common selections compact.

In [ ]:
adata.ap.select(
    obs=[ap.all_of(["batch"]), ap.any_of(["missing", "score"]), ap.last_col()],
    var=ap.starts_with("feature"),
    x=ap.num_range("g", [3, 0]),
)

## Arrange and Slice

`arrange` reorders an AnnData axis, and the slice helpers take position-based subsets.

In [ ]:
adata.ap.arrange(obs=ap.desc("score"), var=ap.col("length"))

In [ ]:
adata.ap.slice_head(n=2)

## Mutate and Summarize

`mutate` writes new `obs` or `var` columns. Matrix, layer, `obsm`, and `varm` values are read-only sources for those assignments.

In [ ]:
adata.ap.mutate(
    obs={"score2": ap.col("score") * 2},
    x={"g0_counts": ap.col("g0")},
    obsm={"X_pca": {"pc0": ap.col("0")}},
    layer="counts",
).obs

In [ ]:
adata.ap.summarize(
    obs={"mean_score": ap.mean("score"), "cells": ap.n()},
    by="batch",
)

## Rename, Relocate, Distinct, and Counts

The metadata-safe dplyr-style verbs return AnnData objects while preserving aligned containers.

In [ ]:
adata.ap.rename(obs={"condition": "batch"}, x={"Gene0": "g0"})

In [ ]:
adata.ap.relocate(obs=["score"], before="batch")

In [ ]:
adata.ap.distinct(obs="batch", keep_all=True)

In [ ]:
adata.ap.add_count(by="batch", name="batch_n").obs

## Group By

`group_by` returns a grouped wrapper. You can iterate over groups, summarize them, count them, and use group-local expressions such as `row_number`.

In [ ]:
grouped = adata.ap.group_by(obs="batch")
grouped.group_keys()

In [ ]:
for key, group in grouped:
    print(key, group.shape)

In [ ]:
grouped.mutate(obs={"within_batch": ap.row_number()}).obs

## Plot-Ready Tables

`to_df` returns a wide pandas table. `to_tidy` returns a long table with one row per observation-feature pair.

In [ ]:
adata.ap.to_df(obs=["batch"], x=["g0", "g3"], obsm={"X_pca": ["0"]})

In [ ]:
adata.ap.to_tidy(obs=["batch"], x=["g0", "g3"])

## Pipe

`pipe` sends the AnnData object into a normal Python function, which makes it useful at the end of a chain.

In [ ]:
def shape_summary(data):
    """Return a compact AnnData shape summary."""
    return {"n_obs": data.n_obs, "n_vars": data.n_vars}


adata.ap.filter(obs=ap.col("batch") == "A").ap.pipe(shape_summary)